In [2]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [3]:
# Datasets  & Dataloaders
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
])

trainset = CIFAR10(root="./data", train=True, download=True, transform = transform)
testset = CIFAR10(root="./data", train = False, download=True, transform = transform)


100%|███████████████████████████████████████████████████████████████| 170498071/170498071 [00:57<00:00, 2964022.59it/s]


Extracting ./data\cifar-10-python.tar.gz to ./data
Files already downloaded and verified


In [4]:
trainloader = DataLoader(trainset, batch_size = 64, shuffle = True)
testloader = DataLoader(testset, batch_size=64)

# Build the CNN

In [5]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            
            nn.Conv2d(64, 128, kernel_size=3, padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2)
        )
        
        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128, 256),
            nn.ReLU(),
            
            nn.Linear(256, 10)
        )
        
    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1)
        x = self.fc_layers(x)
        
        return x        

In [6]:
model = CNN()

In [7]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the Model

In [ ]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=0.0012478060124780212
epoch=2/10 & loss=0.001457461920540656
epoch=3/10 & loss=0.0008650991465429515
epoch=4/10 & loss=0.0008005929724944522
epoch=5/10 & loss=0.0008160368255946947
epoch=6/10 & loss=0.0005771616840606456
epoch=7/10 & loss=0.0005647175376067686
epoch=8/10 & loss=0.0012007119405604994
epoch=9/10 & loss=0.0002617619340986852
epoch=10/10 & loss=0.00030638174632626115


In [ ]:
# Evaluate CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels/total_labels* 100}")

accuracy = 68.75


In [ ]:
# Training + Validation

#Training
model.train()
epochs = 10

for epoch in range(epochs):
    model.train()
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images)
        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    epoch_training_loss = epoch_training_loss / len(trainloader)

    # validation
    model.eval()
    running_val_loss = 0.0

    with torch.no_grad():
        for images, labels in testloader:
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_val_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} | Train Loss: {epoch_training_loss:.4f} | Val Loss: {running_val_loss:.4f}")

epoch=1/10 | Train Loss: 0.1352 | Val Loss: 179.9428
epoch=2/10 | Train Loss: 0.1165 | Val Loss: 200.8176
epoch=3/10 | Train Loss: 0.0953 | Val Loss: 222.9929
epoch=4/10 | Train Loss: 0.0966 | Val Loss: 230.9225
epoch=5/10 | Train Loss: 0.0807 | Val Loss: 242.1705
epoch=6/10 | Train Loss: 0.0747 | Val Loss: 231.4441
epoch=7/10 | Train Loss: 0.0843 | Val Loss: 251.2224
epoch=8/10 | Train Loss: 0.0684 | Val Loss: 283.1278
epoch=9/10 | Train Loss: 0.0684 | Val Loss: 276.6486
epoch=10/10 | Train Loss: 0.0731 | Val Loss: 275.4034
